In [1]:
!pip install tensorflow

In [2]:
import pandas as pd
import numpy as np

from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Embedding, Dense

In [5]:
ls

 Volume in drive C is Windows-SSD
 Volume Serial Number is 1283-82A2

 Directory of C:\Users\Pradheesha\Downloads\Netflix recommendation system

09-04-2026  11:59    <DIR>          .
08-04-2026  08:34    <DIR>          ..
08-04-2026  20:19    <DIR>          .ipynb_checkpoints
09-04-2026  11:37             1,713 app.py
08-04-2026  20:19             6,908 Clustering.ipynb
08-04-2026  08:59            91,106 colloborative_filtering.ipynb
08-04-2026  08:50           102,243 Data_Preprocessing.ipynb
04-04-2026  11:58         1,493,648 movie.csv
05-04-2026  16:06     3,269,638,009 processed_df.csv
04-04-2026  11:59       690,353,377 rating.csv
04-04-2026  11:58        21,725,514 tag.csv
09-04-2026  11:59            10,863 Untitled.ipynb
               9 File(s)  3,983,423,381 bytes
               3 Dir(s)  322,021,134,336 bytes free


In [3]:
df = pd.read_csv("C:\\Users\\Pradheesha\\Downloads\\Netflix recommendation system\\processed_df.csv")

In [5]:
movie_ids = df['movieId'].astype("category").cat.codes

movie_map = dict(zip(df['movieId'], movie_ids))
inv_movie_map = dict(zip(movie_ids, df['movieId']))

df['movie_encoded'] = movie_ids

In [6]:
df = df.sort_values(['userId','timestamp'])

user_sequences = df.groupby('userId')['movie_encoded'].apply(list)

In [7]:
X, y = [], []
max_len = 10

for seq in user_sequences:

    seq = seq[-max_len:]   # limit length

    for i in range(1, len(seq)):
        X.append(seq[:i])
        y.append(seq[i])

In [8]:
X = pad_sequences(X, maxlen=10)
y = np.array(y)

In [9]:
num_movies = df['movie_encoded'].nunique()

In [11]:
model = Sequential([
    Embedding(input_dim=num_movies, output_dim=64),
    LSTM(64),
    Dense(num_movies, activation='softmax')
])

In [12]:
model.compile(
    loss='sparse_categorical_crossentropy',
    optimizer='adam'
)

model.fit(X, y, epochs=3, batch_size=256)

Epoch 1/3
4869/4869 ━━━━━━━━━━━━━━━━━━━━ 605s 123ms/step - loss: 7.3747
Epoch 2/3
4869/4869 ━━━━━━━━━━━━━━━━━━━━ 601s 123ms/step - loss: 6.7425
Epoch 3/3
4869/4869 ━━━━━━━━━━━━━━━━━━━━ 567s 116ms/step - loss: 6.5392


In [14]:
model.save('models/lstm_model.h5')

print("LSTM model saved")

LSTM model saved


In [16]:
import pickle
with open('models/movie_map.pkl', 'wb') as f:
    pickle.dump(movie_map, f)

print("✅ Movie map saved")

✅ Movie map saved
